# Trend and correlation analysis 關鍵字趨勢與關聯分析

A vs. B corelation

        The Pearson correlation coefficient measures the linear association between variables. Its value can be interpreted like so:

        +1 - Complete positive correlation
        +0.8 - Strong positive correlation
        +0.6 - Moderate positive correlation
        0 - no correlation whatsoever  ==>無線性關聯，但不代表沒有關係 
        -0.6 - Moderate negative correlation
        -0.8 - Strong negative correlation
        -1 - Complete negative correlation

        https://stackabuse.com/calculating-pearson-correlation-coefficient-in-python-with-numpy/

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [6]:

x_simple = np.array([1, 2, 3, 4, 5])
y_simple = np.array([11, 12, 15, 14, 15])
# x_simple = np.array([-2, -1, 0, 1, 2])
# y_simple = np.array([4, 1, 3, 2, 0])
rho = np.corrcoef(x_simple, y_simple)

print(rho)


[[1.         0.87038828]
 [0.87038828 1.        ]]


# Load Data 

In [7]:
import pandas as pd
from datetime import datetime, timedelta

df = pd.read_csv('./news_dataset_preprocessed_for_django.csv',sep='|')

In [8]:
df.head(1)

,item_id,title,category,content,link,date,photo_link,tokens_v2,top_keys_freq,summary,sentiment
0,aipl_20220124_1001,殲16D首擾台 專家示警：國軍應強化電子作戰,政治,中共殲16D新型電戰機今天首次擾台，學者及退將分析，殲16D可能已量產並進行實戰化運用，國軍...,https://www.cna.com.tw/news/aipl/202201240357....,2022-01-24,https://imgcdn.cna.com.tw/www/WebPhotos/200/20...,"['中共', '新型', '電戰機', '學者', '退將', '分析', '量產', '進...","[('中共', 7), ('台灣', 7), ('電子', 6), ('美軍', 6), (...","['其中殲16D新型電戰機為首次現蹤', '中共殲16D新型電戰機今天首次擾台', '13架...",0.96


In [9]:
df.shape

(26179, 11)

# All-in-one function: Get time-span frequency

In [10]:
def get_keyword_occurrence_time_series(query_keywords, cond='or', weeks=12):
    # end_date
    end_date = df.date.max()
    # start date
    start_date_delta = (datetime.strptime(end_date, '%Y-%m-%d').date() - timedelta(weeks=weeks)).strftime('%Y-%m-%d')
    start_date_min = df.date.min()
    # set start_date as the larger one from the start_date_delta and start_date_min
    start_date = max(start_date_delta,   start_date_min)

    # (1) proceed filtering: a duration of a period of time
    # 期間條件
    period_condition = (df.date >= start_date) & (df.date <= end_date) 

    # (2) proceed filtering: news category
    # and or 條件
    if (cond == 'and'):
        # query keywords condition使用者輸入關鍵字條件and
        condition = period_condition & df.content.apply(lambda text: all((qk in text) for qk in query_keywords)) #寫法:all()
    else:
        # query keywords condition使用者輸入關鍵字條件
        condition = period_condition & df.content.apply(lambda text: any((qk in text) for qk in query_keywords)) #寫法:any()

    # condiction is a list of True or False boolean value
    df_query = df[condition]

    query_freq = pd.DataFrame({'date_index':pd.to_datetime( df_query.date ),'freq':[1 for _ in range(len(df_query))]})
    print(query_freq)


    # 開始時間、結束時間兩項必須也加入到query_freq，計算次數時才會有完整的時間軸，否則時間軸長度會因為新聞時間不同，導致時間軸不一致
    dt_start_date = datetime.strptime(start_date, '%Y-%m-%d')
    dt_end_date = datetime.strptime(end_date, '%Y-%m-%d')    
    query_freq = pd.concat([query_freq, pd.DataFrame({'date_index': [dt_start_date], 'freq': [0]})])
    query_freq = pd.concat([query_freq, pd.DataFrame({'date_index': [dt_end_date], 'freq': [0]})])
    
    freq_data = query_freq.groupby(pd.Grouper(key='date_index', freq='D')).sum()

    freq_data.reset_index(inplace=True)

    # 只有y, 沒有時間變數x
    y_freq_data = freq_data.freq.to_list()
    # 有時間變數x,y
    line_xy_data = [{'x':date.strftime('%Y-%m-%d'),'y':freq} for date, freq in zip(freq_data.date_index,freq_data.freq)]

    return line_xy_data, y_freq_data


In [12]:
query_keywords = ['肺炎', '冠狀', '疫苗','covid-19','Covid-19']
weeks=12
xy_covid, y_covid = get_keyword_occurrence_time_series(query_keywords)


      date_index  freq
1741  2022-01-29     1
1742  2022-01-29     1
1749  2022-01-29     1
1750  2022-01-29     1
1754  2022-01-29     1
...          ...   ...
26164 2022-04-23     1
26167 2022-04-23     1
26171 2022-04-23     1
26172 2022-04-23     1
26178 2022-04-23     1

[4993 rows x 2 columns]


In [13]:
xy_covid

[{'x': '2022-01-29', 'y': 58},
 {'x': '2022-01-30', 'y': 42},
 {'x': '2022-01-31', 'y': 41},
 {'x': '2022-02-01', 'y': 36},
 {'x': '2022-02-02', 'y': 46},
 {'x': '2022-02-03', 'y': 51},
 {'x': '2022-02-04', 'y': 46},
 {'x': '2022-02-05', 'y': 51},
 {'x': '2022-02-06', 'y': 43},
 {'x': '2022-02-07', 'y': 68},
 {'x': '2022-02-08', 'y': 66},
 {'x': '2022-02-09', 'y': 77},
 {'x': '2022-02-10', 'y': 87},
 {'x': '2022-02-11', 'y': 61},
 {'x': '2022-02-12', 'y': 57},
 {'x': '2022-02-13', 'y': 50},
 {'x': '2022-02-14', 'y': 71},
 {'x': '2022-02-15', 'y': 72},
 {'x': '2022-02-16', 'y': 73},
 {'x': '2022-02-17', 'y': 79},
 {'x': '2022-02-18', 'y': 63},
 {'x': '2022-02-19', 'y': 31},
 {'x': '2022-02-20', 'y': 34},
 {'x': '2022-02-21', 'y': 58},
 {'x': '2022-02-22', 'y': 53},
 {'x': '2022-02-23', 'y': 57},
 {'x': '2022-02-24', 'y': 53},
 {'x': '2022-02-25', 'y': 39},
 {'x': '2022-02-26', 'y': 29},
 {'x': '2022-02-27', 'y': 26},
 {'x': '2022-02-28', 'y': 32},
 {'x': '2022-03-01', 'y': 37},
 {'x': '

In [14]:
y_covid

[58,
 42,
 41,
 36,
 46,
 51,
 46,
 51,
 43,
 68,
 66,
 77,
 87,
 61,
 57,
 50,
 71,
 72,
 73,
 79,
 63,
 31,
 34,
 58,
 53,
 57,
 53,
 39,
 29,
 26,
 32,
 37,
 41,
 41,
 38,
 24,
 37,
 46,
 47,
 42,
 42,
 41,
 29,
 29,
 49,
 49,
 56,
 59,
 47,
 31,
 27,
 53,
 66,
 54,
 49,
 53,
 38,
 51,
 74,
 75,
 91,
 82,
 71,
 44,
 63,
 65,
 65,
 88,
 104,
 83,
 71,
 60,
 78,
 99,
 126,
 97,
 86,
 54,
 64,
 93,
 118,
 105,
 102,
 57,
 52]

In [15]:
query_keywords = ['陳時中']
weeks=12
xy_sccheng, y_scchen = get_keyword_occurrence_time_series(query_keywords)
y_scchen[0:10]


      date_index  freq
1801  2022-01-29     1
1804  2022-01-29     1
1805  2022-01-29     1
1816  2022-01-29     1
1838  2022-01-29     1
...          ...   ...
26071 2022-04-23     1
26078 2022-04-23     1
26083 2022-04-23     1
26090 2022-04-23     1
26137 2022-04-23     1

[743 rows x 2 columns]


[5, 5, 2, 0, 0, 0, 6, 13, 3, 11]

In [16]:
len(y_scchen)

85

In [17]:
np.corrcoef(y_covid, y_scchen)


array([[1.        , 0.75953465],
       [0.75953465, 1.        ]])

In [18]:
from scipy import stats 

In [19]:
stats.pearsonr(y_covid, y_scchen)

PearsonRResult(statistic=0.759534651026398, pvalue=3.5856030005718766e-17)

In [15]:
query_keywords = ['柯文哲', '柯p', '柯P']
weeks=12
xy_kop, y_kop = get_keyword_occurrence_time_series(query_keywords)
y_kop[0:10]


      date_index  freq
1801  2022-01-29     1
1822  2022-01-29     1
2258  2022-02-01     1
2262  2022-02-01     1
2335  2022-02-01     1
...          ...   ...
25808 2022-04-22     1
25812 2022-04-22     1
25838 2022-04-22     1
26057 2022-04-23     1
26065 2022-04-23     1

[233 rows x 2 columns]


[2, 0, 0, 3, 2, 0, 3, 2, 5, 3]

In [16]:
query_keywords = ['台北市','民眾黨']
weeks=12
xy_taipei, y_taipei = get_keyword_occurrence_time_series(query_keywords)
y_taipei[0:10]


      date_index  freq
1740  2022-01-29     1
1741  2022-01-29     1
1756  2022-01-29     1
1797  2022-01-29     1
1801  2022-01-29     1
...          ...   ...
26078 2022-04-23     1
26083 2022-04-23     1
26090 2022-04-23     1
26150 2022-04-23     1
26163 2022-04-23     1

[1697 rows x 2 columns]


[8, 7, 10, 17, 11, 8, 8, 14, 14, 21]

In [17]:
np.corrcoef(y_kop, y_taipei)


array([[1.        , 0.60215177],
       [0.60215177, 1.        ]])

In [20]:
from scipy import stats 
def get_correlation_data(queryA, queryB, weeks=12):
    a_line_xy_data, a_freq_data = get_keyword_occurrence_time_series(queryA,weeks)
    b_line_xy_data, b_freq_data = get_keyword_occurrence_time_series(queryB,weeks)

    try:
        pearson_coef, p_value = stats.pearsonr(a_freq_data, b_freq_data)
        # person = np.corrcoef(y_A, y_B)[0,1]
    except:
        return None
    pearson_coef = round(pearson_coef,3)
    p_value = round(p_value,5)
    return pearson_coef, p_value, a_line_xy_data, b_line_xy_data

In [21]:
queryA = ['肺炎', '冠狀', '疫苗', 'covid-19']
queryB = ['陳時中']
get_correlation_data(queryA, queryB)[:2]


      date_index  freq
1741  2022-01-29     1
1742  2022-01-29     1
1749  2022-01-29     1
1750  2022-01-29     1
1754  2022-01-29     1
...          ...   ...
26164 2022-04-23     1
26167 2022-04-23     1
26171 2022-04-23     1
26172 2022-04-23     1
26178 2022-04-23     1

[4988 rows x 2 columns]
      date_index  freq
1801  2022-01-29     1
1804  2022-01-29     1
1805  2022-01-29     1
1816  2022-01-29     1
1838  2022-01-29     1
...          ...   ...
26071 2022-04-23     1
26078 2022-04-23     1
26083 2022-04-23     1
26090 2022-04-23     1
26137 2022-04-23     1

[743 rows x 2 columns]


(0.76, 0.0)

In [22]:
queryA = ['半導體','IC']
queryB = ['台積','張忠謀']
get_correlation_data(queryA, queryB)[:2]


      date_index  freq
1762  2022-01-29     1
1779  2022-01-29     1
1780  2022-01-29     1
1894  2022-01-29     1
1928  2022-01-30     1
...          ...   ...
25852 2022-04-22     1
25889 2022-04-22     1
25987 2022-04-23     1
26027 2022-04-23     1
26031 2022-04-23     1

[1192 rows x 2 columns]
      date_index  freq
1945  2022-01-30     1
2080  2022-01-31     1
2126  2022-01-31     1
2222  2022-02-01     1
2229  2022-02-01     1
...          ...   ...
25746 2022-04-22     1
25754 2022-04-22     1
25769 2022-04-22     1
25889 2022-04-22     1
25987 2022-04-23     1

[600 rows x 2 columns]


(0.761, 0.0)

In [23]:
from scipy.stats import linregress
a = [15, 12, 8, 8, 7, 7, 7, 6, 5, 3]
b = [10, 25, 17, 11, 13, 17, 20, 13, 9, 15]
linregress(a, b)

LinregressResult(slope=0.20833333333333331, intercept=13.375, rvalue=0.14499815458068518, pvalue=0.689401448116695, stderr=0.5026170462708364, intercept_stderr=4.247039688703678)

In [36]:
# To Convert your lists to pandas data frames convert your lists into pandas dataframes
import pandas as pd
from scipy import stats 
data = {'list 1': [2, 4, 6, 8], 'list 2': [4, 16, 36, 64]}


df_ = pd.DataFrame(data, columns=['list 1', 'list 2'])


# define the columns to perform calculations on
pearson_coef, p_value = stats.pearsonr(df_["list 1"], df_["list 2"])
print("Pearson Correlation Coefficient: ", pearson_coef,
      "and a P-value of:", p_value)  # Results


Pearson Correlation Coefficient:  0.9843740386976971 and a P-value of: 0.01562596130230287


# Do it step by step

### Calculate time duration of news dataset

the duration of data

In [ ]:
df.date.min()

'2022-01-24'

In [ ]:
df.date.max()

'2022-04-23'

In [ ]:
d1 = datetime.strptime(str(df.date.min()), '%Y-%m-%d')
d2 = datetime.strptime(str(df.date.max()), '%Y-%m-%d')

In [ ]:
d1

datetime.datetime(2022, 1, 24, 0, 0)

In [ ]:
date_num = d2 - d1

In [ ]:
date_num

datetime.timedelta(days=89)

In [ ]:
date_num.days

89

In [ ]:
base = datetime.datetime.today()
date_list = [base - datetime.timedelta(days=x) for x in range(numdays)]


In [ ]:
#[d1 - timedelta(days=x) for x in range(date_num.days)]


In [ ]:
import pandas as pd
from datetime import datetime


In [ ]:

pd.date_range(datetime.today(), periods=10).tolist()


[Timestamp('2022-04-29 15:30:41.776958', freq='D'),
 Timestamp('2022-04-30 15:30:41.776958', freq='D'),
 Timestamp('2022-05-01 15:30:41.776958', freq='D'),
 Timestamp('2022-05-02 15:30:41.776958', freq='D'),
 Timestamp('2022-05-03 15:30:41.776958', freq='D'),
 Timestamp('2022-05-04 15:30:41.776958', freq='D'),
 Timestamp('2022-05-05 15:30:41.776958', freq='D'),
 Timestamp('2022-05-06 15:30:41.776958', freq='D'),
 Timestamp('2022-05-07 15:30:41.776958', freq='D'),
 Timestamp('2022-05-08 15:30:41.776958', freq='D')]

In [ ]:

pd.date_range(start=df.date.min(), end=df.date.max())
#pd.date_range(df.date.min(), periods=10).tolist()


DatetimeIndex(['2022-01-24', '2022-01-25', '2022-01-26', '2022-01-27',
               '2022-01-28', '2022-01-29', '2022-01-30', '2022-01-31',
               '2022-02-01', '2022-02-02', '2022-02-03', '2022-02-04',
               '2022-02-05', '2022-02-06', '2022-02-07', '2022-02-08',
               '2022-02-09', '2022-02-10', '2022-02-11', '2022-02-12',
               '2022-02-13', '2022-02-14', '2022-02-15', '2022-02-16',
               '2022-02-17', '2022-02-18', '2022-02-19', '2022-02-20',
               '2022-02-21', '2022-02-22', '2022-02-23', '2022-02-24',
               '2022-02-25', '2022-02-26', '2022-02-27', '2022-02-28',
               '2022-03-01', '2022-03-02', '2022-03-03', '2022-03-04',
               '2022-03-05', '2022-03-06', '2022-03-07', '2022-03-08',
               '2022-03-09', '2022-03-10', '2022-03-11', '2022-03-12',
               '2022-03-13', '2022-03-14', '2022-03-15', '2022-03-16',
               '2022-03-17', '2022-03-18', '2022-03-19', '2022-03-20',
      

In [ ]:
# end_date
end_date = df.date.max()
# start date
start_date = (datetime.strptime(end_date, '%Y-%m-%d').date() -
              timedelta(weeks=12)).strftime('%Y-%m-%d')
# start_date = df.date.min()



In [ ]:
date_sample = pd.date_range(start=start_date, end=end_date)


In [ ]:
pd.DataFrame({'date_index': pd.to_datetime(date_sample),
              'freq': [0 for _ in range(len(date_sample))]})


,date_index,freq
0,2022-01-29,0
1,2022-01-30,0
2,2022-01-31,0
3,2022-02-01,0
4,2022-02-02,0
...,...,...
80,2022-04-19,0
81,2022-04-20,0
82,2022-04-21,0
83,2022-04-22,0


In [ ]:
date_base = pd.DataFrame({'date_index': pd.to_datetime(date_sample),
              'freq': [0 for _ in range(len(date_sample))]})

In [ ]:
date_base.set_index('date_index', inplace=True)


In [ ]:
date_base


,freq
date_index,
2022-01-29,0
2022-01-30,0
2022-01-31,0
2022-02-01,0
2022-02-02,0
...,...
2022-04-19,0
2022-04-20,0
2022-04-21,0


In [ ]:
date_base.index

DatetimeIndex(['2022-01-29', '2022-01-30', '2022-01-31', '2022-02-01',
               '2022-02-02', '2022-02-03', '2022-02-04', '2022-02-05',
               '2022-02-06', '2022-02-07', '2022-02-08', '2022-02-09',
               '2022-02-10', '2022-02-11', '2022-02-12', '2022-02-13',
               '2022-02-14', '2022-02-15', '2022-02-16', '2022-02-17',
               '2022-02-18', '2022-02-19', '2022-02-20', '2022-02-21',
               '2022-02-22', '2022-02-23', '2022-02-24', '2022-02-25',
               '2022-02-26', '2022-02-27', '2022-02-28', '2022-03-01',
               '2022-03-02', '2022-03-03', '2022-03-04', '2022-03-05',
               '2022-03-06', '2022-03-07', '2022-03-08', '2022-03-09',
               '2022-03-10', '2022-03-11', '2022-03-12', '2022-03-13',
               '2022-03-14', '2022-03-15', '2022-03-16', '2022-03-17',
               '2022-03-18', '2022-03-19', '2022-03-20', '2022-03-21',
               '2022-03-22', '2022-03-23', '2022-03-24', '2022-03-25',
      

In [44]:
query_keywords = ['台積電']
query_keywords = ['肺炎']
query_keywords = ['威爾史密斯']
# end_date
end_date = df.date.max()
# start date
start_date = (datetime.strptime(end_date, '%Y-%m-%d').date() -
              timedelta(weeks=12)).strftime('%Y-%m-%d')

# start_date = df.date.min()

# Filtering news
df_query = df[(df['date'] >= start_date) & (df['date'] <= end_date)
              & df['tokens_v2'].str.contains('|'.join(query_keywords))]  # alternative code: apply() and any()

date_samples = df_query.date
# date_samples = pd.date_range(start=start_date, end=df.date.max())


query_freq = pd.DataFrame({'date_index': pd.to_datetime(
    date_samples), 'freq': [1 for _ in range(len(date_samples))]})

# 開始時間、結束時間兩項必須也加入到query_freq，計算次數時才會有完整的時間軸，否則時間軸長度會因為新聞時間不同，導致時間軸不一致
dt_start_date = datetime.strptime(start_date, '%Y-%m-%d')
dt_end_date = datetime.strptime(end_date, '%Y-%m-%d')  

query_freq = pd.concat([query_freq, pd.DataFrame({'date_index': [dt_start_date], 'freq': [0]})])
query_freq = pd.concat([query_freq, pd.DataFrame({'date_index': [dt_end_date], 'freq': [0]})])
      
# query_freq = query_freq.append({'date_index': dt_start_date, 'freq': 0}, ignore_index=True)
# query_freq = query_freq.append({'date_index': dt_end_date, 'freq': 0}, ignore_index=True)

freq_data = query_freq.groupby(pd.Grouper(key='date_index', freq='D')).sum()

freq_data.reset_index(inplace=True)
y_freq = freq_data.freq.to_list()
xy_freq = [{'x':date.strftime('%Y-%m-%d'),'y':freq} for date, freq in zip(freq_data.date_index,freq_data.freq)]


In [45]:
y_freq

[0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 13,
 11,
 3,
 3,
 2,
 3,
 0,
 3,
 0,
 0,
 2,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [46]:
freq_data

,date_index,freq
0,2022-01-29,0
1,2022-01-30,0
2,2022-01-31,0
3,2022-02-01,0
4,2022-02-02,0
...,...,...
80,2022-04-19,0
81,2022-04-20,0
82,2022-04-21,0
83,2022-04-22,0


In [47]:
freq_data.reset_index(inplace=True)
freq_data

,index,date_index,freq
0,0,2022-01-29,0
1,1,2022-01-30,0
2,2,2022-01-31,0
3,3,2022-02-01,0
4,4,2022-02-02,0
...,...,...,...
80,80,2022-04-19,0
81,81,2022-04-20,0
82,82,2022-04-21,0
83,83,2022-04-22,0


In [48]:
freq_data.freq.to_list()

[0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 13,
 11,
 3,
 3,
 2,
 3,
 0,
 3,
 0,
 0,
 2,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [49]:
[{'x':date.strftime('%Y-%m-%d'),'y':freq} for date, freq in zip(freq_data.date_index,freq_data.freq)]

[{'x': '2022-01-29', 'y': 0},
 {'x': '2022-01-30', 'y': 0},
 {'x': '2022-01-31', 'y': 0},
 {'x': '2022-02-01', 'y': 0},
 {'x': '2022-02-02', 'y': 0},
 {'x': '2022-02-03', 'y': 0},
 {'x': '2022-02-04', 'y': 1},
 {'x': '2022-02-05', 'y': 0},
 {'x': '2022-02-06', 'y': 0},
 {'x': '2022-02-07', 'y': 0},
 {'x': '2022-02-08', 'y': 1},
 {'x': '2022-02-09', 'y': 0},
 {'x': '2022-02-10', 'y': 0},
 {'x': '2022-02-11', 'y': 0},
 {'x': '2022-02-12', 'y': 0},
 {'x': '2022-02-13', 'y': 0},
 {'x': '2022-02-14', 'y': 0},
 {'x': '2022-02-15', 'y': 0},
 {'x': '2022-02-16', 'y': 0},
 {'x': '2022-02-17', 'y': 0},
 {'x': '2022-02-18', 'y': 0},
 {'x': '2022-02-19', 'y': 0},
 {'x': '2022-02-20', 'y': 0},
 {'x': '2022-02-21', 'y': 0},
 {'x': '2022-02-22', 'y': 0},
 {'x': '2022-02-23', 'y': 0},
 {'x': '2022-02-24', 'y': 0},
 {'x': '2022-02-25', 'y': 0},
 {'x': '2022-02-26', 'y': 0},
 {'x': '2022-02-27', 'y': 0},
 {'x': '2022-02-28', 'y': 1},
 {'x': '2022-03-01', 'y': 0},
 {'x': '2022-03-02', 'y': 0},
 {'x': '20

In [54]:
end_date = df.date.max()

In [55]:
end_date

'2022-04-23'

In [56]:
# 開始時間、結束時間兩項必須也加入到query_freq，計算次數時才會有完整的時間軸，否則時間軸長度會因為新聞時間不同，導致時間軸不一致
dt_start_date = datetime.strptime(start_date, '%Y-%m-%d')
dt_end_date = datetime.strptime(end_date, '%Y-%m-%d')   

In [57]:
{'date_index': dt_start_date, 'freq': 0}


{'date_index': datetime.datetime(2022, 1, 29, 0, 0), 'freq': 0}

In [58]:
#query_freq.append({'date_index': dt_start_date, 'freq': 0}, ignore_index=True)
#query_freq.append({'date_index': dt_end_date, 'freq': 0}, ignore_index=True)
query_freq = pd.concat([query_freq, pd.DataFrame({'date_index': [dt_start_date], 'freq': [0]})])
query_freq = pd.concat([query_freq, pd.DataFrame({'date_index': [dt_end_date], 'freq': [0]})])


In [ ]:
freq_data = query_freq.groupby(pd.Grouper(key='date_index', freq='D')).sum()


In [ ]:
freq_data


,freq
date_index,
2022-01-29,0
2022-01-30,0
2022-01-31,0
2022-02-01,0
2022-02-02,0
...,...
2022-04-19,0
2022-04-20,0
2022-04-21,0


# views.py in Django

In [ ]:
from django.http import JsonResponse
from django.shortcuts import render
import pandas as pd

def load_data_correlation():
    # Read data from csv file
    global df
    df = pd.read_csv('./news_dataset_preprocessed_for_django.csv',sep='|')

# load data
load_data_correlation()

def home(request):
    return render(request,'app_correlation/home.html')


# csrf_exempt is used for POST
# 單獨指定這一支程式忽略csrf驗證
@csrf_exempt
def api_get_corr_data(request):

    # (1) get keywords, category, condition, and weeks passed from frontend
    userkey1 = request.POST['userkey1']
    userkey2 = request.POST['userkey2']
    print(userkey2)
    print(type(userkey2))

    userkey1 = userkey1.split()
    userkey2 = userkey2.split()

    pearson_coef, p_value, a_line_xy_data, b_line_xy_data = get_correlation_data(userkey1,
        userkey2, weeks=12)

    response = {
        'pearson_coef': pearson_coef,
        'p_value': p_value,
        'a_line_xy_data': a_line_xy_data,
        'b_line_xy_data': b_line_xy_data
    }
    return JsonResponse(response)


def get_correlation_data(queryA, queryB, weeks=12):
    a_line_xy_data, a_freq_data = get_keyword_occurrence_time_series(
        queryA, weeks)
    b_line_xy_data, b_freq_data = get_keyword_occurrence_time_series(
        queryB, weeks)

    try:
        pearson_coef, p_value = stats.pearsonr(a_freq_data, b_freq_data)
        # person = np.corrcoef(y_A, y_B)[0,1]
    except:
        return None

    pearson_coef = round(pearson_coef,3)
    p_value = round(p_value,5)

    return pearson_coef, p_value, a_line_xy_data, b_line_xy_data

def get_keyword_occurrence_time_series(query_keywords, cond='or', weeks=12):
    # end_date
    end_date = df.date.max()
    # start date
    start_date_delta = (datetime.strptime(end_date, '%Y-%m-%d').date() - timedelta(weeks=weeks)).strftime('%Y-%m-%d')
    start_date_min = df.date.min()
    # set start_date as the larger one from the start_date_delta and start_date_min
    start_date = max(start_date_delta,   start_date_min)

    # (1) proceed filtering: a duration of a period of time
    # 期間條件
    period_condition = (df.date >= start_date) & (df.date <= end_date) 

    # (2) proceed filtering: news category
    # and or 條件
    if (cond == 'and'):
        # query keywords condition使用者輸入關鍵字條件and
        condition = period_condition & df.content.apply(lambda text: all((qk in text) for qk in query_keywords)) #寫法:all()
    else:
        # query keywords condition使用者輸入關鍵字條件
        condition = period_condition & df.content.apply(lambda text: any((qk in text) for qk in query_keywords)) #寫法:any()

    # condiction is a list of True or False boolean value
    df_query = df[condition]

    query_freq = pd.DataFrame({'date_index':pd.to_datetime( df_query.date ),'freq':[1 for _ in range(len(df_query))]})


    # 開始時間、結束時間兩項必須也加入到query_freq，計算次數時才會有完整的時間軸，否則時間軸長度會因為新聞時間不同，導致時間軸不一致
    dt_start_date = datetime.strptime(start_date, '%Y-%m-%d')
    dt_end_date = datetime.strptime(end_date, '%Y-%m-%d')    
    
    query_freq = pd.concat([query_freq, pd.DataFrame({'date_index': [dt_start_date], 'freq': [0]})])
    query_freq = pd.concat([query_freq, pd.DataFrame({'date_index': [dt_end_date], 'freq': [0]})])
    
    #query_freq = query_freq.append({'date_index': dt_start_date, 'freq': 0}, ignore_index=True)
    #query_freq = query_freq.append({'date_index': dt_end_date, 'freq': 0}, ignore_index=True)

    freq_data = query_freq.groupby(pd.Grouper(key='date_index', freq='D')).sum()

    freq_data.reset_index(inplace=True)

    # 只有y, 沒有時間變數x
    y_freq_data = freq_data.freq.to_list()
    # 有時間變數x,y
    line_xy_data = [{'x':date.strftime('%Y-%m-%d'),'y':freq} for date, freq in zip(freq_data.date_index,freq_data.freq)]

    return line_xy_data, y_freq_data


print('app_correlation was loaded!')

app_correlation was loaded!
